## 手动实现MoE

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


In [2]:
class BasicExpert(nn.Module):
    def __init__(self, feature_in, feature_out):
        super().__init__()
        self.fc = nn.Linear(feature_in, feature_out)

    def forward(self, x):
        return self.fc(x)
    

In [15]:
class BasicMoE(nn.Module):
    def __init__(self, feature_in, feature_out, num_expert):
        super().__init__()
        self.gate = nn.Linear(feature_in, num_expert)
        ## shape (batch_size, num_expert)
        self.experts = nn.ModuleList(
            BasicExpert(
                feature_in=feature_in,
                feature_out=feature_out
            ) for _ in range(num_expert)
        )

    def forward(self, x):
        ## X shape is (batch, feature_in) # feature_in 是输入的维度
        ## feature_in 也可以叫做hidden_dim hidden_size
        experts_weights = self.gate(x)
        experts_out_list = [
            expert(x) for expert in self.experts
        ]## 每一个expert输出一个(batch, feature_out)

        experts_outputs = [expert_out.unsqueeze(1) for expert_out in experts_out_list]
        ## 希望 expert out 是(b, 1, feature_out)
        expert_out = torch.cat(
            experts_outputs,
            dim=1
        )

        # expert_out shape : (b, num_experts, feature_out)
        
        # expert_weights
        experts_weights = F.softmax(experts_weights, dim=1)


        # expert_weights shape : (b, num_experts)

        experts_weight = experts_weights.unsqueeze(1)

        output = experts_weight @ expert_out
        
        output = output.squeeze(1)

        # 希望的输出 (batch, feature_out)

        return output
    

x = torch.rand(4, 512)

basic_moe = BasicMoE(512, 128, 4)
output = basic_moe(x)
print(output.shape)


torch.Size([4, 128])


In [43]:
@dataclass
class MOEConfig:
    hidden_dim: int = 128
    expert_number: int = 10
    top_k: int = 3
    shared_experts_number: int = 5


In [ ]:
class MoERouter(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate = nn.Linear(config.hidden_dim, config.expert_number)
        self.expert_number = config.expert_number
        self.hidden_dim = config.hidden_dim
        self.top_k = config.top_k
    
    def forward(self, x):  ## x shape is (batch_size * seq_len, hidden_dim)
        router_logits = self.gate(x) ## (batch_size * seq_len, num_expert)
        router_probs = F.softmax(router_logits, dim=-1, dtype=torch.float) 
        ## 然后选出来topk个专家的输出
        router_weights, select_experts_index = torch.topk(router_probs, dim=-1,k=self.top_k)  ## 这个是可以反向传播的
        ## (batch_size, seq_len, k)
        router_weights = router_weights / router_weights.sum(
            dim=-1, keepdim=True
        )
        ## 重新归一化

        router_weights = router_weights.to(x.dtype)

        experts_mask = F.one_hot(
            select_experts_index,
            num_classes=self.expert_number
        )

        experts_mask = experts_mask.permute(2, 1, 0)


        return router_logits, router_weights, select_experts_index, experts_mask
    

In [ ]:
moe_router = MoERouter(config=MOEConfig())

demo_x = torch.rand(8, 128)

demo_y = moe_router(demo_x)


In [61]:
class SparseMoE(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.experts = nn.ModuleList(
            BasicExpert(
                config.hidden_dim,
                config.hidden_dim,
            ) for _ in range(config.expert_number)
        )
        self.router = MoERouter(config=config) # TODO

    def forward(self, x):
        # X shape (batch, seq_len, hidden_dim)
        batch_size, seq_len, hidden_dim = x.size()

        ## 需要对token维度进行计算 需要对x reshape成(batch * seq_len, hidden_dim)
        hidden_states = x.view(-1, hidden_dim)

        ## 做相关的专家的计算
        router_logits, router_weights, select_index, expert_mask = self.router(
            hidden_states
        )
        ## 这里出来的shape是 (expert_number, top_k, batch * seq_len)
        
        ## 最终肯定是(batch_size * seq_len, hidden_dim)
        final_hidden_states = torch.zeros(
            (batch_size * seq_len, hidden_dim),
            dtype=hidden_states.dtype,
            device=hidden_states.device
        )
        
        ## 把选中这个专家的token的hidden_states加到final_hidden_states中

        for expert_index in range(self.config.expert_number):
            expert_layer = self.experts[expert_index]

            ## expert_mask (expert_number, top_k, batch * seq_len)
            current_expert_mask = expert_mask[expert_index]

            index, top_x = torch.where(current_expert_mask)

            # index 是0 or 1 假设topk是2
            # 表示这个token是作为当前专家的top1还是top2
            
            # top_x是token在batch * seq_len中的位置索引
            # 例如对于batch_size = 2，seq_le = 4的输入：
            # topx的取值范围就是0-7，表示在展平后第八个token中的位置(都是一个一维的值)
            ## idx是用来选weight的
            ## topx是用来选hidden_states
            current_state = hidden_states.unsqueeze(
                0
            )[:, top_x, :].reshape(-1, hidden_dim)
            current_state = expert_layer(current_state)
            ## current_state shape is (selected_token_number, hidden_dim)
            # 100个token选中了
            current_token_router_weight = router_weights[top_x, index]
            current_token_router_weight = current_token_router_weight.unsqueeze(-1)
            ## currten_token_router_weight本来的shape是(selected_token_number)
            ## 经过unsqueeze之后变成了(selected_token_number, 1)的shape
            current_hidden_sates = current_state * current_token_router_weight
            ## 这里就是通过广播，变成了(selected_token_number, hidden_dim)
            final_hidden_states.index_add_(
                0,
                top_x,
                current_hidden_sates.to(hidden_states.dtype)
            )
        # 把final_hidden_shape还原到原来的shape
        final_hidden_states = final_hidden_states.reshape(batch_size, seq_len, hidden_dim)

        return final_hidden_states, router_logits



def test_demo():
    x = torch.rand(2, 4, 128)
    config = MOEConfig()
    sparseMoE = SparseMoE(config=config)
    demo_y = sparseMoE(x)
    print(demo_y[0].shape, demo_y[1].shape)


test_demo()

torch.Size([2, 4, 128]) torch.Size([8, 10])


In [81]:
class SharedExpertMoE(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.router_experts_moe = SparseMoE(config)
        self.shared_experts_moe = nn.ModuleList(
            [
                BasicExpert(self.config.hidden_dim, self.config.hidden_dim) for _ in range(self.config.shared_experts_number)
            ]
        )
    
    def forward(self, x):
         batch_size, seq_len, hidden_dim = x.size()

         shared_expert_outputs_list = [expert_item(x) for expert_item in self.shared_experts_moe]
         shared_expert_outputs_list_uns = [
             expert_item.unsqueeze(1) for expert_item in shared_expert_outputs_list
             ]
         shared_expert_output = torch.cat(
             shared_expert_outputs_list_uns,
             dim=1
             )
         shared_expert_output_sum = shared_expert_output.sum(dim=1)

         sparse_expert_output = self.router_experts_moe(x)

         final_expert_output = shared_expert_output_sum + sparse_expert_output[0]

         return final_expert_output, sparse_expert_output[1]



def test_demo():
    x = torch.rand(2, 4, 128)
    config = MOEConfig()
    sharedMoE = SharedExpertMoE(config=config)
    demo_y = sharedMoE(x)
    print(demo_y[0].shape, demo_y[1].shape)


test_demo()

torch.Size([2, 4, 128]) torch.Size([8, 10])
